# Notebook 8 — International Marketing Campaign Scheduling with IP2Location.io

**Use case:** Plan marketing campaign delivery around local working hours using IP geolocation and time zones. This notebook **creates a schedule**; it does **not send emails or notifications**.

You will load an audience CSV, enrich unique public IPs using the [IP2Location.io API](https://www.ip2location.io/ip2location-documentation), choose an appropriate **IANA/Olson time zone**, calculate the next local weekday send time, and export marketing-ready reports.

> **Important:** IP geolocation is approximate. The sample IP-to-country associations in **demo mode are intentionally synthetic**. A real IP may be anycast, proxied, or geographically different. Respect permission and consent before contacting users.

## 1. Setup and requirements

Install dependencies in your notebook environment if needed:

```python
%pip install pandas requests matplotlib tzdata
```

`tzdata` is useful on Windows, where system IANA time-zone data may be missing. Uses Python 3.9+ (`zoneinfo`).

**Recommended workflow:** Download `sample_campaign_audiences.csv` into the same directory as this notebook. If it is absent, the notebook creates a copy of the included sample automatically. Start with `MODE = "demo"` (no network). Switch to `"live"` after you are satisfied with the example.

In [ ]:
from pathlib import Path
from datetime import datetime, date, time, timedelta, timezone
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import ipaddress
import os
import re
import time as time_module

import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

print('Python dependencies loaded successfully.')

## 2. Shared configuration

**All constants used in either mode are initialized here or in the cells below without a demo-only dependency.** Run cells in order, or choose *Kernel → Restart & Run All* when changing modes.

- In `demo`, all lookups use clearly labeled simulated records.
- In `live`, unique public IP addresses are queried with optional Bearer token authentication. Set an environment variable (`IP2LOCATION_API_KEY`); do not commit credentials to the notebook.
- `PLANNING_START_UTC` is a fixed reference point for reproducible **demo** output. **Live mode automatically uses the current UTC time** when executed.
- A scheduled send is allowed only at the configured local hour, on a weekday, within the planning horizon, for opted-in rows.

In [ ]:
MODE = 'demo'  # 'demo' or 'live'
SAMPLE_COLUMNS = ['audience_id','visitor_ip','campaign','preferred_send_hour','timezone_override','opted_in']
INPUT_CSV = Path('sample_campaign_audiences.csv')
OUTPUT_DIR = Path('outputs_notebook_08')
API_URL = 'https://api.ip2location.io/'
API_KEY = os.getenv('IP2LOCATION_API_KEY', '').strip()
PLANNING_START_UTC = '2026-10-27T00:00:00+00:00'  # deterministic example
MAX_UNIQUE_IP_LOOKUPS = 25
REQUEST_TIMEOUT_SECONDS = 15
MAX_RETRIES = 2
RETRY_DELAY_SECONDS = 1.0
MIN_LEAD_MINUTES = 60
PLANNING_HORIZON_DAYS = 14
LOCAL_BUSINESS_HOUR_START = 9
LOCAL_BUSINESS_HOUR_END_EXCLUSIVE = 17

assert MODE in {'demo', 'live'}
assert 0 < LOCAL_BUSINESS_HOUR_START < LOCAL_BUSINESS_HOUR_END_EXCLUSIVE <= 24
START_UTC = (datetime.now(timezone.utc) if MODE == 'live'
             else datetime.fromisoformat(PLANNING_START_UTC.replace('Z', '+00:00')))
if START_UTC.tzinfo is None:
    raise ValueError('PLANNING_START_UTC must include a time-zone offset.')
START_UTC = START_UTC.astimezone(timezone.utc)
print(f'Mode: {MODE} | UTC planning start: {START_UTC.isoformat()}')

## 3. Sample audience data and synthetic demo fixtures

These simulated locations demonstrate multiple time zones, including DST-observing zones. **Demo fixture country/city names are not claims about the real public IPs.** Some rows intentionally contain invalid/private IPs, missing fixture data, an invalid time-zone override, a non-business send hour, and missing consent.

`timezone_override` is for a **confirmed** audience-specific IANA time zone. It takes priority over the API zone when valid; a sample override is illustrative and should be verified before production use.

In [ ]:
SAMPLE_ROWS = [
    ('A001', '8.8.8.8', 'Product Launch', '10', '', 'yes'),
    ('A002', '8.8.8.8', 'Newsletter', '11', 'America/New_York', 'yes'),
    ('A003', '1.1.1.1', 'Product Launch', '10', '', 'yes'),
    ('A004', '9.9.9.9', 'Newsletter', '9', '', 'yes'),
    ('A005', '208.67.222.222', 'Product Launch', '10', 'Asia/Kuala_Lumpur', 'yes'),
    ('A006', '8.8.4.4', 'Newsletter', '11', '', 'yes'),
    ('A007', '1.0.0.1', 'Product Launch', '10', '', 'yes'),
    ('A008', '149.112.112.112', 'Newsletter', '10', '', 'yes'),
    ('A009', '64.6.64.6', 'Product Launch', '10', '', 'yes'),
    ('A010', '80.80.80.80', 'Newsletter', '10', '', 'yes'),
    ('A011', '94.140.14.14', 'Product Launch', '10', '', 'yes'),
    ('A012', '1.1.1.1', 'Newsletter', '15', 'Europe/London', 'yes'),
    ('A013', '9.9.9.9', 'Product Launch', '11', 'Asia/Tokyo', 'yes'),
    ('A014', '8.8.8.8', 'Product Launch', '9', '', 'no'),
    ('A015', '192.168.1.15', 'Newsletter', '10', '', 'yes'),
    ('A016', 'bad-ip-value', 'Newsletter', '10', '', 'yes'),
    ('A017', '1.0.0.1', 'Newsletter', '10', 'Mars/Olympus', 'yes'),
    ('A018', '94.140.14.14', 'Newsletter', '10', '', 'no'),
    ('A019', '4.2.2.2', 'Product Launch', '10', '', 'yes'),
    ('A020', '208.67.222.222', 'Newsletter', '18', '', 'yes'),
]

# All geolocations below are synthetic fixtures for *offline demonstration only*.
DEMO_GEO = {
    '8.8.8.8': ('US','United States','New York','New York','America/New_York','-04:00'),
    '1.1.1.1': ('GB','United Kingdom','England','London','Europe/London','+00:00'),
    '9.9.9.9': ('JP','Japan','Tokyo','Tokyo','Asia/Tokyo','+09:00'),
    '208.67.222.222': ('MY','Malaysia','Kuala Lumpur','Kuala Lumpur','Asia/Kuala_Lumpur','+08:00'),
    '8.8.4.4': ('DE','Germany','Berlin','Berlin','Europe/Berlin','+01:00'),
    '1.0.0.1': ('AU','Australia','New South Wales','Sydney','Australia/Sydney','+11:00'),
    '149.112.112.112': ('CA','Canada','Ontario','Toronto','America/Toronto','-04:00'),
    '64.6.64.6': ('SG','Singapore','Singapore','Singapore','Asia/Singapore','+08:00'),
    '80.80.80.80': ('BR','Brazil','São Paulo','São Paulo','America/Sao_Paulo','-03:00'),
    '94.140.14.14': ('AE','United Arab Emirates','Dubai','Dubai','Asia/Dubai','+04:00'),
}

if not INPUT_CSV.exists():
    pd.DataFrame(SAMPLE_ROWS, columns=SAMPLE_COLUMNS).to_csv(INPUT_CSV, index=False)
    print(f'Created sample input: {INPUT_CSV}')
else:
    print(f'Using existing input: {INPUT_CSV}')

In [ ]:
audience = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
required = {'audience_id', 'visitor_ip', 'campaign', 'preferred_send_hour', 'opted_in'}
missing = required - set(audience.columns)
if missing:
    raise ValueError(f'Missing required CSV columns: {sorted(missing)}')
if 'timezone_override' not in audience.columns:
    audience['timezone_override'] = ''
for col in SAMPLE_COLUMNS:
    audience[col] = audience[col].astype(str).str.strip()
if audience['audience_id'].duplicated().any():
    raise ValueError('audience_id must be unique for review and export.')

def parse_opt_in(value):
    return str(value).strip().lower() in {'yes','true','1','y'}

audience['has_consent'] = audience['opted_in'].map(parse_opt_in)
display(audience.head(10))
print(f'Rows: {len(audience)} | opted in: {int(audience.has_consent.sum())}')

## 4. IP address validation

Only **globally routable** IPv4/IPv6 addresses are eligible for API lookup. A consent check runs before requesting the IP geolocation. Multiple audience rows sharing an IP require only one lookup.

In [ ]:
def classify_ip(value):
    try:
        obj = ipaddress.ip_address(str(value).strip())
        return 'public' if obj.is_global else 'non_public'
    except ValueError:
        return 'invalid'

audience['ip_status'] = audience['visitor_ip'].map(classify_ip)
audience['eligible_lookup'] = audience['has_consent'] & audience['ip_status'].eq('public')
print(audience.groupby(['ip_status', 'has_consent']).size().to_string())
unique_eligible_ips = audience.loc[audience.eligible_lookup, 'visitor_ip'].drop_duplicates().tolist()
print('Unique eligible IPs:', len(unique_eligible_ips))
print('Maximum requests allowed in this run:', MAX_UNIQUE_IP_LOOKUPS)

## 5. API client and response normalization

The [standard IP Geolocation API](https://www.ip2location.io/ip2location-documentation) returns `time_zone` as a UTC offset. Higher plans can also return `time_zone_info.olson` (e.g., `America/New_York`), which is the DST-aware form required for future scheduling. This notebook will **not convert a fixed UTC offset into a guessed IANA zone**.

- Uses one HTTPS call per unique eligible IP (up to a configurable cap).
- Stops on authentication/quota exhaustion.
- Handles network and API errors without exposing the API key.
- No outgoing network calls in demo mode.

In [ ]:
GEO_COLUMNS = ['country_code','country_name','region_name','city_name','api_iana_zone','api_utc_offset','lookup_status','lookup_error']
SESSION = requests.Session()

def normalize_geo_payload(payload):
    tz_info = payload.get('time_zone_info') or {}
    if not isinstance(tz_info, dict):
        tz_info = {}
    return {
        'country_code': str(payload.get('country_code') or ''),
        'country_name': str(payload.get('country_name') or ''),
        'region_name': str(payload.get('region_name') or ''),
        'city_name': str(payload.get('city_name') or ''),
        'api_iana_zone': str(tz_info.get('olson') or ''),
        'api_utc_offset': str(payload.get('time_zone') or ''),
        'lookup_status': 'ok',
        'lookup_error': '',
    }

def failed_lookup(status, message=''):
    result = {field: '' for field in GEO_COLUMNS}
    result['lookup_status'] = status
    result['lookup_error'] = message
    return result

class StopFurtherLookups(Exception):
    pass

def lookup_ip_live(ip, session=SESSION):
    headers = {'Accept': 'application/json'}
    if API_KEY:
        headers['Authorization'] = f'Bearer {API_KEY}'
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(API_URL, params={'ip': ip, 'format': 'json'}, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.status_code in (401, 403, 429):
                raise StopFurtherLookups(f'HTTP {response.status_code}: authorization or quota/rate limit')
            if response.status_code >= 500 and attempt < MAX_RETRIES:
                time_module.sleep(RETRY_DELAY_SECONDS * (attempt + 1))
                continue
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict):
                return failed_lookup('api_error', 'Unexpected non-object JSON response')
            if payload.get('error'):
                error = payload['error']
                err_code = error.get('error_code') if isinstance(error, dict) else ''
                if str(err_code) == '10000':
                    raise StopFurtherLookups('API key invalid or query quota exhausted (10000)')
                return failed_lookup('api_error', f'API error code {err_code}')
            return normalize_geo_payload(payload)
        except StopFurtherLookups:
            raise
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt == MAX_RETRIES:
                return failed_lookup('network_error', 'Request timed out or connection failed')
            time_module.sleep(RETRY_DELAY_SECONDS * (attempt + 1))
        except (requests.exceptions.RequestException, ValueError):
            return failed_lookup('api_error', 'HTTP or JSON response error')
    return failed_lookup('api_error', 'Exhausted attempts')

def lookup_ip_demo(ip):
    if ip not in DEMO_GEO:
        return failed_lookup('demo_not_mapped', 'No synthetic fixture for this IP')
    cc, country, region, city, tzid, offset = DEMO_GEO[ip]
    data = normalize_geo_payload({
        'country_code': cc, 'country_name': country, 'region_name': region,
        'city_name': city, 'time_zone': offset, 'time_zone_info': {'olson': tzid}
    })
    data['lookup_status'] = 'demo_simulated'
    return data

## 6. Run deduplicated lookups

The lookup budget applies to **unique eligible IP addresses**, not input rows. Failed lookups and records missing an IANA zone remain visible in the review queue rather than disappearing.

In [ ]:
lookup_cache = {}
lookup_stopped_reason = ''
for index, ip in enumerate(unique_eligible_ips):
    if index >= MAX_UNIQUE_IP_LOOKUPS:
        lookup_cache[ip] = failed_lookup('skipped_query_cap', 'Configured lookup cap reached')
        continue
    if lookup_stopped_reason:
        lookup_cache[ip] = failed_lookup('lookup_not_attempted', lookup_stopped_reason)
        continue
    if MODE == 'demo':
        lookup_cache[ip] = lookup_ip_demo(ip)
    else:
        try:
            lookup_cache[ip] = lookup_ip_live(ip)
        except StopFurtherLookups as exc:
            lookup_stopped_reason = str(exc)
            lookup_cache[ip] = failed_lookup('quota_or_auth_error', lookup_stopped_reason)

print(f'Unique lookups evaluated: {len(lookup_cache)}')
if lookup_stopped_reason:
    print('Stopped further API requests:', lookup_stopped_reason)
print(pd.Series([r['lookup_status'] for r in lookup_cache.values()]).value_counts().to_string())

In [ ]:
geo_rows = []
for row in audience.itertuples(index=False):
    if not row.has_consent:
        geo = failed_lookup('excluded_no_consent', 'Audience has not opted in')
    elif row.ip_status != 'public':
        geo = failed_lookup('invalid_or_nonpublic_ip', row.ip_status)
    else:
        geo = lookup_cache.get(row.visitor_ip, failed_lookup('lookup_not_attempted'))
    geo_rows.append(geo)

enriched = pd.concat([audience.reset_index(drop=True), pd.DataFrame(geo_rows, columns=GEO_COLUMNS)], axis=1)
display(enriched[['audience_id','visitor_ip','country_name','api_iana_zone','api_utc_offset','lookup_status']].head(12))

## 7. Choose a DST-aware scheduling time zone

Precedence: **confirmed `timezone_override` → API `time_zone_info.olson` → needs review**.

A bare offset such as `-04:00` only describes a particular UTC offset; it cannot predict DST transitions. Therefore the notebook deliberately blocks future scheduling for an offset-only record. Users of the Free or Starter plan can populate `timezone_override` with a **verified visitor preference or other trusted IANA zone**.

An invalid override is recorded as a warning. If a valid API IANA zone exists, it is used instead. The notebook does not schedule invalid/private IPs or non-consenting rows.

In [ ]:
def valid_iana(value):
    value = str(value or '').strip()
    if not value:
        return False
    # Require a named IANA zone rather than an offset; UTC is also a valid named zone.
    if value != 'UTC' and '/' not in value:
        return False
    try:
        ZoneInfo(value)
        return True
    except (ZoneInfoNotFoundError, ValueError):
        return False

def choose_zone(row):
    if not row['has_consent']:
        return {'selected_timezone': '', 'timezone_source': '', 'timezone_warning': '', 'planning_status': 'excluded_no_consent'}
    if row['ip_status'] != 'public':
        return {'selected_timezone': '', 'timezone_source': '', 'timezone_warning': '', 'planning_status': 'invalid_or_nonpublic_ip'}
    if row['lookup_status'] not in ('ok','demo_simulated'):
        return {'selected_timezone': '', 'timezone_source': '', 'timezone_warning': '', 'planning_status': 'geolocation_unavailable'}
    override = row['timezone_override']
    api_zone = row['api_iana_zone']
    warning = 'Invalid manual IANA time-zone override' if override and not valid_iana(override) else ''
    if valid_iana(override):
        return {'selected_timezone': override, 'timezone_source': 'confirmed_override', 'timezone_warning': '', 'planning_status': 'ready'}
    if valid_iana(api_zone):
        return {'selected_timezone': api_zone, 'timezone_source': 'api_olson', 'timezone_warning': warning, 'planning_status': 'ready'}
    return {'selected_timezone': '', 'timezone_source': '', 'timezone_warning': warning or 'API returned only a UTC offset or no time zone', 'planning_status': 'needs_iana_timezone'}

zone_selection = pd.DataFrame([choose_zone(row) for _, row in enriched.iterrows()])
enriched = pd.concat([enriched.reset_index(drop=True), zone_selection], axis=1)
print(enriched['planning_status'].value_counts().to_string())

## 8. Plan the next eligible local weekday send

Each opted-in, geolocated row is assigned the next weekday at its chosen local hour within the planning horizon. Hours outside `[09:00, 17:00)` are treated as configuration errors and sent to review; they are not silently changed. `ZoneInfo` computes the correct UTC offset **at the future send date**, including future DST transitions.

This is a **planning report**, not a mailer. Production workflows should also enforce suppression lists, sender local-hour policies, regional legal requirements, and a final consent check immediately before delivery.

In [ ]:
def plan_one(row):
    result = {'scheduled_local': '', 'scheduled_utc': '', 'utc_send_hour': '', 'send_weekday': '', 'schedule_status': ''}
    if row['planning_status'] != 'ready':
        result['schedule_status'] = row['planning_status']
        return result
    try:
        local_hour = int(str(row['preferred_send_hour']).strip())
    except ValueError:
        result['schedule_status'] = 'invalid_send_hour'
        return result
    if not (LOCAL_BUSINESS_HOUR_START <= local_hour < LOCAL_BUSINESS_HOUR_END_EXCLUSIVE):
        result['schedule_status'] = 'outside_business_hours'
        return result
    tz = ZoneInfo(row['selected_timezone'])
    now_local = START_UTC.astimezone(tz)
    for day_delta in range(PLANNING_HORIZON_DAYS + 1):
        candidate_day = now_local.date() + timedelta(days=day_delta)
        if candidate_day.weekday() >= 5:
            continue
        candidate_local = datetime.combine(candidate_day, time(hour=local_hour), tzinfo=tz)
        candidate_utc = candidate_local.astimezone(timezone.utc)
        if candidate_utc < START_UTC + timedelta(minutes=MIN_LEAD_MINUTES):
            continue
        if candidate_utc.astimezone(tz).replace(tzinfo=None) != candidate_local.replace(tzinfo=None):
            # Avoid nonexistent local times in nonstandard DST transitions.
            continue
        result.update({
            'scheduled_local': candidate_local.isoformat(timespec='minutes'),
            'scheduled_utc': candidate_utc.isoformat(timespec='minutes'),
            'utc_send_hour': candidate_utc.hour,
            'send_weekday': candidate_local.strftime('%A'),
            'schedule_status': 'scheduled'
        })
        return result
    result['schedule_status'] = 'no_eligible_day_in_horizon'
    return result

plans = pd.DataFrame([plan_one(row) for _, row in enriched.iterrows()])
schedule = pd.concat([enriched.reset_index(drop=True), plans], axis=1)
print(schedule['schedule_status'].value_counts().to_string())

## 9. Inspect the campaign plan

The time shown for each visitor is the **local time where the audience is scheduled**, alongside the matching UTC time used by automation systems. A missing schedule always carries an explanation in `schedule_status`.

In [ ]:
display(schedule[[
    'audience_id','campaign','country_name','selected_timezone','timezone_source',
    'scheduled_local','scheduled_utc','schedule_status','timezone_warning'
]].head(20))

## 10. Analyze campaign markets and the UTC dispatch timeline

A market-level view shows how many audience records are scheduled or need attention. Since input records may represent audiences rather than individual people, counts reflect **CSV rows**, not deduplicated people.

In [ ]:
report = schedule.copy()
report['market'] = report['country_name'].replace('', 'Unknown')
report['is_scheduled'] = report['schedule_status'].eq('scheduled')
report['is_review'] = report['has_consent'] & ~report['is_scheduled']
market_summary = (report.groupby(['market','campaign'], dropna=False)
                  .agg(audience_rows=('audience_id','count'), consented_rows=('has_consent','sum'),
                       scheduled_rows=('is_scheduled','sum'), review_rows=('is_review','sum'))
                  .reset_index().sort_values(['market','campaign']))
market_summary['schedule_rate_pct'] = (100 * market_summary['scheduled_rows'] / market_summary['consented_rows'].where(market_summary['consented_rows'] != 0)).round(1)
display(market_summary)

review_queue = schedule[schedule['schedule_status'] != 'scheduled'].copy()
print('Records needing review (includes excluded/non-consenting rows):', len(review_queue))
display(review_queue[['audience_id','campaign','visitor_ip','schedule_status','lookup_status','timezone_warning']])

In [ ]:
scheduled_only = schedule[schedule['schedule_status'].eq('scheduled')].copy()
if len(scheduled_only):
    hourly_utc = (scheduled_only.groupby(['scheduled_utc','campaign']).size()
                  .reset_index(name='audience_rows').sort_values(['scheduled_utc','campaign']))
else:
    hourly_utc = pd.DataFrame(columns=['scheduled_utc','campaign','audience_rows'])
display(hourly_utc.head(12))

In [ ]:
country_plot = (report.groupby('market')['is_scheduled'].sum().sort_values(ascending=False))
fig, ax = plt.subplots(figsize=(10, 4.5))
country_plot.plot(kind='bar', ax=ax)
ax.set(title='Scheduled audience records by market', xlabel='Market', ylabel='Records')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
if not scheduled_only.empty:
    scheduled_only['send_utc_hour_bucket'] = pd.to_datetime(scheduled_only['scheduled_utc'], utc=True).dt.hour
    utc_hour_counts = scheduled_only.groupby('send_utc_hour_bucket').size().reindex(range(24), fill_value=0)
    utc_hour_counts.plot(kind='bar', ax=ax)
else:
    utc_hour_counts = pd.Series(0, index=range(24))
    utc_hour_counts.plot(kind='bar', ax=ax)
ax.set(title='Planned sends by UTC hour (all scheduled dates combined)', xlabel='UTC hour', ylabel='Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. DST sanity check

The two local send times below are both **10:00 in New York**, but the UTC hour changes after the early-November US daylight-saving transition. A bare offset from an October API response cannot capture that change for a November campaign.

In [ ]:
new_york = ZoneInfo('America/New_York')
for day in [date(2026, 10, 30), date(2026, 11, 2)]:
    local_dt = datetime.combine(day, time(10, 0), tzinfo=new_york)
    print(f'{local_dt.isoformat()} -> {local_dt.astimezone(timezone.utc).isoformat()}')
assert datetime(2026,10,30,10,tzinfo=new_york).astimezone(timezone.utc).hour == 14
assert datetime(2026,11,2,10,tzinfo=new_york).astimezone(timezone.utc).hour == 15

## 12. Export the marketing schedule and reports

The exports contain audience IDs and IPs, so treat them as sensitive operational data and use access controls. The notebook will write files into `outputs_notebook_08/` relative to the notebook's working directory.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    'campaign_schedule.csv': schedule,
    'campaign_review_queue.csv': review_queue,
    'campaign_market_summary.csv': market_summary,
    'campaign_utc_dispatch.csv': hourly_utc,
}
for filename, table in paths.items():
    filepath = OUTPUT_DIR / filename
    table.to_csv(filepath, index=False)
    print(f'{filepath} ({len(table)} rows)')

## 13. Self-checks and production notes

**Important distinctions:**

- `time_zone` from the basic API is a current UTC offset. An offset alone is **not enough** to schedule safely across future DST changes; records need a confirmed IANA zone (manual override or `time_zone_info.olson`).
- Geo-IP is an estimate, **not a person's precise location**. Anycast DNS IPs in this tutorial illustrate code flow, not actual customer distributions.
- Consent is checked before API enrichment and before scheduling. Do not use the planning CSV as a substitute for your sending platform's final permission checks.
- This notebook does not send messages, estimate deliverability, or claim an optimal hour based on user engagement. You can replace `preferred_send_hour` with a measured optimal hour from your own analytics later.
- Free and Starter plans may not include the comprehensive time-zone object; the review queue tells you which rows need a verified `timezone_override`.

**Live mode:** set `MODE = "live"` (the planning start then uses current UTC), set an API key in your environment if applicable, verify the input CSV, and use **Restart & Run All**. Be mindful of per-plan API limits.

In [ ]:
assert len(schedule) == len(audience)
assert schedule['audience_id'].is_unique
assert not schedule.loc[~schedule.has_consent, 'schedule_status'].eq('scheduled').any()
assert not schedule.loc[schedule.ip_status.ne('public'), 'schedule_status'].eq('scheduled').any()
assert len(lookup_cache) == len(unique_eligible_ips)
for _, item in schedule[schedule.schedule_status.eq('scheduled')].iterrows():
    local = datetime.fromisoformat(item.scheduled_local)
    utc = datetime.fromisoformat(item.scheduled_utc)
    assert local.weekday() < 5
    assert LOCAL_BUSINESS_HOUR_START <= local.hour < LOCAL_BUSINESS_HOUR_END_EXCLUSIVE
    assert local.astimezone(timezone.utc) == utc.astimezone(timezone.utc)
    assert utc >= START_UTC + timedelta(minutes=MIN_LEAD_MINUTES)
assert int(market_summary.audience_rows.sum()) == len(audience)
assert int(market_summary.scheduled_rows.sum()) == int(schedule.schedule_status.eq('scheduled').sum())
print('All scheduling invariants and export reconciliation checks passed.')